In [ ]:
from uw_sim.audio_simulator import AudioSimulator, DataSet
import pathlib
import yaml

config = yaml.safe_load(pathlib.Path("../config.yaml").read_text())
PATHS = config["paths"]
backgrounds = pathlib.Path(PATHS["background"])
events = pathlib.Path(PATHS["events"])
masks = pathlib.Path(PATHS["masks"])
output = pathlib.Path(PATHS["output"])
dataset_output = pathlib.Path(PATHS["datasets"])
denoise_output = pathlib.Path(PATHS["denoised"])

## Initialise the simulator and set the parameters

In [ ]:




simulator = AudioSimulator(
    background_folder=backgrounds,
    events_folder=events,
    mask_folder=masks,
    output_folder=output,
    sample_rate=48000,
    duration=10,
)
output_audio, metadata_file = simulator.simulate_audio(snr=10, num_events=3)
print(f"Output audio file: {output_audio}")
print(f"Metadata file: {metadata_file}")

# Example usage of DataSet
dataset = DataSet(
    background_folder=backgrounds,
    events_folder=events,
    mask_folder=masks,
    output_folder=output,
    snr_values=config["dataset"]["snr_values"],
    files_per_snr=config["dataset"]["num_files_per_snr"],
    file_length=config["dataset"]["duration"],
    sample_rate=config["dataset"]["samplerate"],
    events_per_file=config["dataset"]["num_events_per_file"],
)

## Generate the dataset

In [ ]:
dataset.generate()
dataset.generate_dataframe()
df = dataset.dataframe
if df is None:
    raise RuntimeError("Dataset dataframe was not created.")
print(df.head())

In [ ]:
import pandas as pd
df = pd.read_pickle(dataset_output / config["dataset"]["dataframe_name"])
df.head()

In [ ]:
import importlib
import uw_sim.util as util

util = importlib.reload(util)

df_bacp, df_denoised = util.write_bacpipe_annotations(
    new_name
)
df_bacp.head(36).audiofilename.values
df_denoised.head(25).audiofilename.values

In [ ]:
# Quick validation/summary of the generated dataset
df["num_events"] = df["event_files"].apply(len)
df["has_events"] = df["num_events"] > 0

snr_summary = (
    df.groupby("snr")
    .agg(
        total_files=("uuid", "count"),
        files_with_events=("has_events", "sum"),
        avg_num_events=("num_events", "mean"),
    )
    .assign(event_ratio=lambda x: x["files_with_events"] / x["total_files"])
    .sort_index()
)

display(snr_summary)

class_summary = (
    df[df["has_events"]]
    .explode("event_classes")
    .groupby(["snr", "event_classes"])
    .size()
    .unstack(fill_value=0)
)

display(class_summary)

In [ ]:
df = pd.read_pickle(dataset_output / config["dataset"]["dataframe_name"])


In [ ]:
df.drop(columns=["mask"], inplace=True)

In [ ]:
name = 'light_'+ config["dataset"]["dataframe_name"]
new_name = dataset_output / name

df.to_pickle(new_name)

In [ ]:
pd.read_pickle(new_name).head()

In [ ]:
from uw_sim import util
import importlib
util = importlib.reload(util)

DF_PATH = dataset_output / 'light_20260319_dataset.pkl'
print (f"Dataframe path: {DF_PATH}")
print (f"Denoised path: {denoise_output}")

output_df, output_denoised_df = util.write_bacpipe_annotations(
    dataframe_path=DF_PATH,
    denoised_path=denoise_output,
    buffer=1
)

In [ ]:
output_denoised_df[["audiofilename"]].values[34]